In [ ]:
!pip install -q datasets

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from datasets import load_dataset
from scipy.sparse import hstack
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, classification_report

keras.utils.set_random_seed(42)
print(tf.config.list_physical_devices("GPU"))

In [ ]:
quora_pairs = load_dataset("nyu-mll/glue", "qqp")
full_train_frame = quora_pairs["train"].to_pandas()
test_frame = quora_pairs["validation"].to_pandas()
print(full_train_frame.shape, test_frame.shape)
display(full_train_frame.head(10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
full_train_frame["label"].map({0: "different", 1: "duplicate"}).value_counts().plot(kind="bar", ax=axes[0], rot=0, color=["steelblue", "darkorange"])
axes[0].set_title("Label balance")
question_lengths = pd.concat([full_train_frame["question1"], full_train_frame["question2"]]).str.split().str.len()
axes[1].hist(question_lengths.clip(upper=80), bins=80, color="gray")
axes[1].set_title("Words per question")
plt.show()
print("95th percentile question length:", question_lengths.quantile(0.95))

In [ ]:
contraction_map = {
    "what's": "what is", "won't": "will not", "can't": "cannot", "n't": " not", "i'm": "i am", "'re": " are",
    "'s": " ", "'ve": " have", "'ll": " will", "'d": " would"
}

def clean_text(text):
    text = str(text).lower()
    text = text.replace("’", "'")
    for short_form, long_form in contraction_map.items():
        text = text.replace(short_form, long_form)
    text = re.sub(r"(\d+)k\b", r"\g<1>000", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

for frame in [full_train_frame, test_frame]:
    frame["clean_q1"] = frame["question1"].map(clean_text)
    frame["clean_q2"] = frame["question2"].map(clean_text)

for original, cleaned in zip(full_train_frame["question1"][:5], full_train_frame["clean_q1"][:5]):
    print(original, "->", cleaned)

In [ ]:
train_frame, val_frame = train_test_split(full_train_frame, test_size=0.05, random_state=42, stratify=full_train_frame["label"])
train_labels = train_frame["label"].values.astype("float32")
val_labels = val_frame["label"].values.astype("float32")
test_labels = test_frame["label"].values.astype("float32")
print(len(train_frame), len(val_frame), len(test_frame))

In [ ]:
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=3, sublinear_tf=True)
tfidf.fit(pd.concat([train_frame["clean_q1"], train_frame["clean_q2"]]))

def tfidf_pair_features(frame):
    first = tfidf.transform(frame["clean_q1"])
    second = tfidf.transform(frame["clean_q2"])
    return hstack([abs(first - second), first.multiply(second)]).tocsr()

baseline_classifier = LogisticRegression(max_iter=1000, C=2.0)
baseline_classifier.fit(tfidf_pair_features(train_frame), train_labels)
baseline_test_scores = baseline_classifier.predict_proba(tfidf_pair_features(test_frame))[:, 1]
print(f"tf-idf baseline  accuracy {accuracy_score(test_labels, baseline_test_scores > 0.5):.4f}  f1 {f1_score(test_labels, baseline_test_scores > 0.5):.4f}  auc {roc_auc_score(test_labels, baseline_test_scores):.4f}")

In [ ]:
max_tokens = 40000
max_length = 30

vectorizer = layers.TextVectorization(max_tokens=max_tokens, output_sequence_length=max_length, standardize=None)
vectorizer.adapt(tf.data.Dataset.from_tensor_slices(pd.concat([train_frame["clean_q1"], train_frame["clean_q2"]]).values).batch(4096))

def to_sequences(texts):
    return vectorizer(tf.constant(texts.values)).numpy()

train_first, train_second = to_sequences(train_frame["clean_q1"]), to_sequences(train_frame["clean_q2"])
val_first, val_second = to_sequences(val_frame["clean_q1"]), to_sequences(val_frame["clean_q2"])
test_first, test_second = to_sequences(test_frame["clean_q1"]), to_sequences(test_frame["clean_q2"])

vocabulary = vectorizer.get_vocabulary()
print("vocabulary size:", len(vocabulary))
print(train_frame["clean_q1"].iloc[0])
print(train_first[0])
print([vocabulary[token] for token in train_first[0] if token != 0])

In [ ]:
class AbsoluteDifference(layers.Layer):
    def call(self, inputs):
        return tf.abs(inputs[0] - inputs[1])

embedding_dim = 200
first_question = keras.Input(shape=(max_length,), dtype="int64", name="first_question")
second_question = keras.Input(shape=(max_length,), dtype="int64", name="second_question")

shared_embedding = layers.Embedding(max_tokens, embedding_dim, mask_zero=True, name="word_embedding")
shared_encoder = layers.Bidirectional(layers.LSTM(96), name="question_encoder")
first_vector = shared_encoder(shared_embedding(first_question))
second_vector = shared_encoder(shared_embedding(second_question))

pair_features = layers.Concatenate()([
    first_vector,
    second_vector,
    AbsoluteDifference()([first_vector, second_vector]),
    layers.Multiply()([first_vector, second_vector])
])
x = layers.Dense(256, activation="relu")(pair_features)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation="relu")(x)
duplicate_probability = layers.Dense(1, activation="sigmoid")(x)

similarity_model = keras.Model([first_question, second_question], duplicate_probability, name="siamese_bilstm")
similarity_model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="binary_crossentropy", metrics=["accuracy", keras.metrics.AUC(name="auc")])
similarity_model.summary()

In [ ]:
similarity_history = similarity_model.fit(
    [train_first, train_second], train_labels,
    validation_data=([val_first, val_second], val_labels),
    epochs=8,
    batch_size=512,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1)
    ]
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 4.5))
for axis, metric in zip(axes, ["loss", "accuracy", "auc"]):
    axis.plot(similarity_history.history[metric], label="train")
    axis.plot(similarity_history.history[f"val_{metric}"], label="validation")
    axis.set_title(metric)
    axis.set_xlabel("epoch")
    axis.legend()
plt.show()

In [ ]:
def symmetric_scores(first, second):
    forward = similarity_model.predict([first, second], batch_size=1024, verbose=0).flatten()
    backward = similarity_model.predict([second, first], batch_size=1024, verbose=0).flatten()
    return (forward + backward) / 2

val_scores = symmetric_scores(val_first, val_second)
thresholds = np.arange(0.2, 0.8, 0.01)
val_f1_scores = [f1_score(val_labels, val_scores > threshold) for threshold in thresholds]
best_threshold = float(thresholds[int(np.argmax(val_f1_scores))])
print(f"best threshold on validation: {best_threshold:.2f}")

test_scores = symmetric_scores(test_first, test_second)
test_predictions = (test_scores > best_threshold).astype(int)
print(f"siamese BiLSTM  accuracy {accuracy_score(test_labels, test_predictions):.4f}  f1 {f1_score(test_labels, test_predictions):.4f}  auc {roc_auc_score(test_labels, test_scores):.4f}")
print(classification_report(test_labels, test_predictions, target_names=["different", "duplicate"], digits=4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for name, scores in [("tf-idf + logistic regression", baseline_test_scores), ("siamese BiLSTM", test_scores)]:
    false_positive_rate, true_positive_rate, _ = roc_curve(test_labels, scores)
    axes[0].plot(false_positive_rate, true_positive_rate, label=f"{name} (auc {roc_auc_score(test_labels, scores):.3f})")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set_xlabel("false positive rate")
axes[0].set_ylabel("true positive rate")
axes[0].set_title("ROC curve on test pairs")
axes[0].legend()
sns.heatmap(confusion_matrix(test_labels, test_predictions), annot=True, fmt="d", cmap="Blues", xticklabels=["different", "duplicate"], yticklabels=["different", "duplicate"], ax=axes[1])
axes[1].set_xlabel("predicted")
axes[1].set_ylabel("actual")
axes[1].set_title("Siamese BiLSTM confusion matrix")
plt.show()

In [ ]:
def similarity_score(first_text, second_text):
    first = vectorizer(tf.constant([clean_text(first_text)])).numpy()
    second = vectorizer(tf.constant([clean_text(second_text)])).numpy()
    return float(symmetric_scores(first, second)[0])

question_pairs = [
    ("How do I learn Python quickly?", "What is the fastest way to learn Python?"),
    ("How do I lose weight?", "What are good ways to gain weight?"),
    ("What is the capital of India?", "Which city is the capital of India?"),
    ("How can I improve my English speaking?", "How do I become fluent in spoken English?"),
    ("Why is the sky blue?", "How do airplanes fly?")
]
for first_text, second_text in question_pairs:
    score = similarity_score(first_text, second_text)
    verdict = "duplicate" if score > best_threshold else "different"
    print(f"{score:.3f}  {verdict:<9}  {first_text} | {second_text}")

In [ ]:
test_frame["score"] = test_scores
test_frame["predicted"] = test_predictions
wrong_pairs = test_frame[test_frame["predicted"] != test_frame["label"]].sample(10, random_state=1)
display(wrong_pairs[["question1", "question2", "label", "score"]])

similarity_model.save_weights("quora_similarity.weights.h5")